In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df = spark.read.table("deliveries.bronze.gen_customers")
df.limit(5).display()

In [0]:
for field in df.schema.fields:
    display(field.dataType)
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
df.groupBy("customer_id").count().where(F.col("count") > 1).display()

In [0]:
df.select("customer_type").distinct().display()

In [0]:
df.select("city").distinct().display()

In [0]:
df = df.withColumn("city", 
              F.when(F.upper(F.col("city")).rlike("^[KC]+.*R.*A.*[KC]+.*O.*W.*$"), "Krakow")
              .when(F.upper(F.col("city")) == "WARSAW", "Warszawa")
              .when(F.col("city").isNull(), "Unknown")
              .otherwise(df.city)) #.select("city").distinct().display()

In [0]:
silver_df = df.select("customer_id", "customer_name", "customer_type", "city")
silver_df.write.option("overwrite", "true").format("delta").saveAsTable("deliveries.silver.gen_customers")